# 03 — Split HUC12 Polygons at Gauge Drainage Boundaries

**Purpose:** For each gauged HUC12, fetch the NLDI upstream drainage basin and
compare it to the HUC12 boundary. If the gauge's basin covers less than 95% of
the HUC12 area, the polygon is **split into two parts**:

| Part | Geometry | HUC_12 code | Routes to (`HU_12_DS`) |
|------|----------|-------------|------------------------|
| Inside gauge basin | intersection | original 12-digit code | `original + "0"` |
| Outside gauge basin | difference | `original + "0"` | original downstream HUC |

**Why 95%?**  
Perfect alignment between NLDI basin polygons and WBD HUC12 boundaries is not
expected. A 5% tolerance absorbs minor edge misalignments while still catching
cases where a meaningful portion of the HUC12 bypasses the gauge.

**Note on 13-digit codes:**  
The `original + "0"` code (e.g., `"0709000102060"`) is a **local convention**
for the remainder polygon. It is not a standard NHD identifier.

**Input:** `cache/nb02_intermediate.pkl`  
**Output:** `cache/nb03_intermediate.pkl`

In [ ]:
# =============================================================================
# USER SETTINGS
# =============================================================================

# Fraction of HUC12 area that must be covered by the gauge basin to skip splitting.
# HUC12s where the gauge basin covers < SPLIT_THRESHOLD of the polygon area are split.
SPLIT_THRESHOLD = 0.95

# Must match the CACHE_DIR used in notebook 02
CACHE_DIR = "./cache"

# =============================================================================
import os
import sys
import pickle
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from pynhd import NLDI

warnings.filterwarnings('ignore')
%matplotlib inline

# Load intermediate outputs from notebook 02
with open(os.path.join(CACHE_DIR, 'nb02_intermediate.pkl'), 'rb') as f:
    nb02 = pickle.load(f)

UMRB_HUC12             = nb02['UMRB_HUC12']
UMRB_HUC12_with_sites  = nb02['UMRB_HUC12_with_sites']
UMRB_HUC12_gauged      = nb02['UMRB_HUC12_gauged']
filtered_site_loc      = nb02['filtered_site_loc']
CACHE_DIR              = nb02.get('CACHE_DIR', CACHE_DIR)

print(f'Gauged HUC12s to process : {len(UMRB_HUC12_gauged)}')
print(f'Split threshold          : {SPLIT_THRESHOLD:.0%}')

## 1. Fetch NLDI drainage basins for all gauged HUC12s

In [ ]:
nldi = NLDI()

# Fetch all gauge basins in one batched NLDI call
Basins = (
    nldi.get_basins(
        feature_ids=list(UMRB_HUC12_gauged['site_no']),
        fsource='nwissite'
    )
    .reset_index()
    .to_crs(UMRB_HUC12_gauged.crs)
    .rename(columns={'identifier': 'site_no'})
)

print(f'Basins fetched : {len(Basins)}')
print(f'CRS match      : {Basins.crs == UMRB_HUC12_gauged.crs}')

## 2. Define the splitting function

In [ ]:
def split_huc12_at_basin_boundary(row, threshold: float = SPLIT_THRESHOLD) -> list[dict]:
    """
    Given a merged row with a HUC12 polygon (geometry_x) and the corresponding
    gauge basin polygon (geometry_y), return one or two row dicts depending on
    whether a split is warranted.

    Parameters
    ----------
    row : Series
        A row from the HUC12 × basin merged GeoDataFrame.
    threshold : float
        If coverage >= threshold, no split is performed.

    Returns
    -------
    list of dict
        One dict (no split) or two dicts (intersection + difference).
    """
    huc12_geom     = row['geometry_x']
    basin_geom     = row['geometry_y']
    original_area  = row['SHAPE_Area']
    original_acres = row['ACRES']

    intersection = huc12_geom.intersection(basin_geom)
    difference   = huc12_geom.difference(basin_geom)
    inter_area   = intersection.area
    diff_area    = difference.area
    coverage     = inter_area / original_area

    if coverage < threshold:
        # --- Split case ---
        # The intersection part stays associated with the gauge.
        # The difference part (outside the gauge basin) is assigned a synthetic
        # 13-digit code (original + '0') and routes to the original downstream HUC.
        return [
            {
                **row,
                'geometry'  : intersection,
                'HUC_12'    : row['HUC_12'],
                'HU_12_DS'  : f"{row['HUC_12']}0",   # routes to the split node
                'SHAPE_Area': inter_area,
                'ACRES'     : original_acres * coverage,
            },
            {
                **row,
                'geometry'  : difference,
                'HUC_12'    : f"{row['HUC_12']}0",   # synthetic remainder code
                'HU_12_DS'  : row['HU_12_DS'],        # routes to original downstream
                'SHAPE_Area': diff_area,
                'ACRES'     : original_acres * (diff_area / original_area),
            },
        ]
    else:
        # --- No split: gauge basin covers the HUC12 adequately ---
        return [{
            **row,
            'geometry'  : huc12_geom,
            'SHAPE_Area': original_area,
            'ACRES'     : original_acres,
        }]

## 3. Apply splitting to all gauged HUC12s

In [ ]:
merged = pd.merge(UMRB_HUC12_gauged, Basins, on='site_no', how='inner')

new_rows = []
n_split  = 0

for _, row in merged.iterrows():
    result = split_huc12_at_basin_boundary(row)
    new_rows.extend(result)
    if len(result) == 2:
        n_split += 1

print(f'HUC12s processed : {len(merged)}')
print(f'HUC12s split     : {n_split}')
print(f'Output rows      : {len(new_rows)}')

In [ ]:
# Build output GeoDataFrame and clean up merge-generated duplicate columns
split_gdf = gpd.GeoDataFrame(new_rows, crs=UMRB_HUC12_gauged.crs)
drop_cols = (
    [c for c in split_gdf.columns if c.endswith(('_right', '_left'))]
    + [c for c in ['geometry_x', 'geometry_y'] if c in split_gdf.columns]
)
split_gdf = split_gdf.drop(columns=drop_cols)

# Clear gauge attributes from synthetic remainder polygons (13-digit codes)
site_cols = [
    'Unnamed: 0', 'site_no', 'station_nm', 'huc_cd', 'site_tp_cd',
    'lat_va', 'long_va', 'dec_lat_va', 'dec_long_va',
    'drain_area_va', 'contrib_drain_area_va'
]
existing_site_cols = [c for c in site_cols if c in split_gdf.columns]
split_gdf = split_gdf.copy()  # avoid SettingWithCopyWarning
split_gdf.loc[
    split_gdf['HUC_12'].astype(str).str.len() == 13,
    existing_site_cols
] = np.nan

print(split_gdf[['HUC_12', 'HU_12_DS', 'site_no', 'SHAPE_Area']].head(10))

## 4. Merge split polygons back with un-gauged HUC12s

In [ ]:
ungauged = UMRB_HUC12_with_sites[
    UMRB_HUC12_with_sites['site_no'].isna()
].reset_index(drop=True)

UMRB_HUC12_with_sites_new = gpd.GeoDataFrame(
    pd.concat([ungauged, split_gdf], ignore_index=True),
    crs=UMRB_HUC12_with_sites.crs
)
print(f'Total HUC12 records in merged network: {len(UMRB_HUC12_with_sites_new)}')

## 5. Visual QC: inspect a split HUC12

In [ ]:
# Find the first HUC12 that was split and plot both parts
split_huc_codes = [
    row['HUC_12'] for _, row in merged.iterrows()
    if len(split_huc12_at_basin_boundary(row)) == 2
]

if split_huc_codes:
    example_huc  = split_huc_codes[0]
    example_site = merged[merged['HUC_12'] == example_huc]['site_no'].iloc[0]

    part_a   = UMRB_HUC12_with_sites_new[UMRB_HUC12_with_sites_new['HUC_12'] == example_huc]
    part_b   = UMRB_HUC12_with_sites_new[UMRB_HUC12_with_sites_new['HUC_12'] == f'{example_huc}0']
    context  = UMRB_HUC12_with_sites[UMRB_HUC12_with_sites['HU_12_DS'] == example_huc]
    site_pt  = filtered_site_loc[filtered_site_loc['site_no'] == example_site]
    bounds   = pd.concat([context, part_a, part_b]).total_bounds

    fig, ax = plt.subplots(figsize=(9, 8))
    UMRB_HUC12_with_sites.plot(ax=ax, color='lightgrey', edgecolor='white', linewidth=0.3)
    context.plot(ax=ax, color='lightblue', edgecolor='black', alpha=0.5, label='Upstream HUC12s')
    part_a.plot(ax=ax, color='red', edgecolor='black', alpha=0.7, label='Inside gauge basin')
    if not part_b.empty:
        part_b.plot(ax=ax, color='orange', edgecolor='black', alpha=0.7, label='Outside gauge basin (remainder)')
    site_pt.plot(ax=ax, color='yellow', markersize=60, zorder=5, label='Gauge site')

    ax.set_xlim(bounds[0], bounds[2])
    ax.set_ylim(bounds[1], bounds[3])
    ax.set_title(f'Split HUC12: {example_huc}', fontsize=13)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.legend(loc='upper right', fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print('No HUC12s were split at the current threshold.')

## 6. Save for notebook 04

In [ ]:
cache_pkl = os.path.join(CACHE_DIR, 'nb03_intermediate.pkl')
with open(cache_pkl, 'wb') as f:
    pickle.dump({
        'UMRB_HUC12_with_sites'    : UMRB_HUC12_with_sites,
        'UMRB_HUC12_with_sites_new': UMRB_HUC12_with_sites_new,
        'filtered_site_loc'        : filtered_site_loc,
        'CACHE_DIR'                : CACHE_DIR,
    }, f)
print(f'Saved: {cache_pkl}')